In [5]:
import torch
!nvidia-smi
torch.cuda.is_available()
TABPFN_TOKEN="tabpfn_sk_oIa-pBgg3FDBGrM95bfSAh-18ypWp8FCvhQ_Bzg3nec"

Mon Sep 21 19:36:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             28W /   70W |     743MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
!pip install tabdpt
!pip install tabpfn
!pip install tabicl
!pip install xgboost
!pip install catboost
!pip install -q gdown

In [7]:
%cd /content
!rm -rf TFM*
!git clone https://github.com/BanafshehKarimian/TFM_BENCH.git
%cd TFM_BENCH

/content
Cloning into 'TFM_BENCH'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 120 (delta 58), reused 111 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 18.68 KiB | 1.25 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/TFM_BENCH


In [9]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np
from tfmbench.datasets import BaseTabularDataset as TabularDataset
from tfmbench.evaluate import evaluate
from tfmbench.benchmark import benchmark_models
from tfmbench.datasets.talent import load_talent_dataset

In [ ]:
X, y = load_breast_cancer(
    return_X_y=True
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)


data = TabularDataset(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    task="classification",
    name="breast_cancer",
)

In [ ]:
results = benchmark_models(
    models=["xgboost", "catboost", "tabicl_v1", "tabicl_v1.1", "tabicl_v2", "tabpfn_v2", "tabpfn_v2.5", "tabpfn_v2.6", "tabpfn_v3", "tabpfn_v3.5", "tabpfn_v3.5_fast"],
    data=data,
    device="cuda",
    tabpfn_token=TABPFN_TOKEN,
    model_kwargs={
        "xgboost": {
            "n_estimators": 500,
            "max_depth": 8,
        },

        "catboost": {
            "iterations": 500,
            "depth": 8,
        },

        "tabdpt_v1.3": {
            "context_size": 8192,
            "n_ensembles": 1,
        },

        "tabpfn_v3": {
            "n_estimators": 8,
        },
    },
)

Running xgboost...


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [19:11:59] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Running catboost...
Running tabicl_v1...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Running tabicl_v1.1...
Running tabicl_v2...
Running tabpfn_v2...
Running tabpfn_v2.5...
Running tabpfn_v2.6...
Running tabpfn_v3...
Running tabpfn_v3.5...
Running tabpfn_v3.5_fast...


In [15]:
results

,model,status,n_train,n_test,n_features,fit_seconds,predict_seconds,total_seconds,peak_gpu_memory_mb,accuracy,balanced_accuracy,f1_macro,roc_auc,log_loss
0,xgboost,ok,455,114,30,0.309059,0.010542,0.319602,0.000000,0.956140,0.945437,0.952129,0.993386,0.089578
1,catboost,ok,455,114,30,41.916075,0.001189,41.917264,0.000000,0.964912,0.957341,0.961911,0.995040,0.090842
2,tabicl_v1,ok,455,114,30,10.550316,0.543359,11.093674,495.378906,0.956140,0.960317,0.953534,0.995370,0.074929
3,tabicl_v1.1,ok,455,114,30,0.525374,0.263291,0.788665,495.253906,0.973684,0.974206,0.971863,0.995701,0.080199
4,tabicl_v2,ok,455,114,30,0.811125,0.452431,1.263556,545.274414,0.973684,0.974206,0.971863,0.996032,0.069909
5,tabpfn_v2,ok,455,114,30,0.459489,0.828062,1.287551,127.383301,0.964912,0.967262,0.962660,0.994048,0.079922
6,tabpfn_v2.5,ok,455,114,30,0.484526,0.535084,1.019610,82.762207,0.973684,0.974206,0.971863,0.996032,0.069347
7,tabpfn_v2.6,ok,455,114,30,0.453937,0.511135,0.965072,82.836426,0.973684,0.974206,0.971863,0.995040,0.085428
8,tabpfn_v3,ok,455,114,30,1.431662,0.744336,2.175997,301.918945,0.982456,0.981151,0.981151,0.996693,0.067783
9,tabpfn_v3.5,ok,455,114,30,2.513539,0.761493,3.275032,931.970215,0.982456,0.981151,0.981151,0.997024,0.073204


In [52]:
for model in ["tabicl_v1", "tabicl_v1.1", "tabicl_v2"]:
    print(model)
    
    result = evaluate(
        model_name=model,
        data=data,
        device="cuda",
    )


    print(result.metrics)
    print(result.fit_seconds)
    print(result.predict_seconds)
    print(result.peak_gpu_memory_mb)

tabicl_v1
{'accuracy': 0.956140350877193, 'balanced_accuracy': np.float64(0.9603174603174602), 'f1_macro': 0.9535338713621913, 'roc_auc': np.float64(0.9953703703703703), 'log_loss': 0.07492859502336296}
0.4109984959995927
0.34087583200016525
495.25390625
tabicl_v1.1
{'accuracy': 0.9736842105263158, 'balanced_accuracy': np.float64(0.9742063492063492), 'f1_macro': 0.9718634306869601, 'roc_auc': np.float64(0.9957010582010583), 'log_loss': 0.08019862081244734}
0.40137645100003283
0.2711546980008279
495.25390625
tabicl_v2
{'accuracy': 0.9736842105263158, 'balanced_accuracy': np.float64(0.9742063492063492), 'f1_macro': 0.9718634306869601, 'roc_auc': np.float64(0.996031746031746), 'log_loss': 0.06990852526418204}
0.4672340630004328
0.4366382789994532
545.2744140625


In [6]:
for model in ["tabpfn_v2", "tabpfn_v2.5", "tabpfn_v2.6", "tabpfn_v3", "tabpfn_v3.5", "tabpfn_v3.5_fast"]:
    print(model)
    
    result = evaluate(
        model_name=model,
        data=data,
        device="cuda",
        tabpfn_token = "tabpfn_sk_oIa-pBgg3FDBGrM95bfSAh-18ypWp8FCvhQ_Bzg3nec",
    )


    print(result.metrics)
    print(result.fit_seconds)
    print(result.predict_seconds)
    print(result.peak_gpu_memory_mb)

tabpfn_v2


tabpfn-v2-classifier-finetuned-zk73skhh.(…): reconstructing file:   0%|          |  0.00B / 29.0MB            

tabpfn-v2-classifier-finetuned-zk73skhh.(…): downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

{'accuracy': 0.9649122807017544, 'balanced_accuracy': np.float64(0.9672619047619048), 'f1_macro': 0.9626596790042581, 'roc_auc': np.float64(0.9940476190476191), 'log_loss': 0.07992157777712897}
11.741694423999434
1.1392986299997574
127.38330078125
tabpfn_v2.5


...

{'accuracy': 0.9824561403508771, 'balanced_accuracy': np.float64(0.9811507936507937), 'f1_macro': 0.9811507936507937, 'roc_auc': np.float64(0.9966931216931217), 'log_loss': 0.06778312458088327}
4.61221532700074
0.792897035000351
301.9189453125
tabpfn_v3.5
{'accuracy': 0.9824561403508771, 'balanced_accuracy': np.float64(0.9811507936507937), 'f1_macro': 0.9811507936507937, 'roc_auc': np.float64(0.9970238095238095), 'log_loss': 0.07320414333746837}
2.4328222580006695
0.6343684539988317
931.97021484375
tabpfn_v3.5_fast


tabpfn-v3.5-fast-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  334MB            

tabpfn-v3.5-fast-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

{'accuracy': 0.9824561403508771, 'balanced_accuracy': np.float64(0.9811507936507937), 'f1_macro': 0.9811507936507937, 'roc_auc': np.float64(0.9966931216931216), 'log_loss': 0.06716286736533611}
4.995144136999443
0.26770195500103
414.75732421875


In [ ]:
for model in ["xgboost", "catboost"]:
    print(model)
    
    result = evaluate(
        model_name=model,
        data=data,
        device="cuda",
    )


    print(result.metrics)
    print(result.fit_seconds)
    print(result.predict_seconds)
    print(result.peak_gpu_memory_mb)

xgboost


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [18:27:00] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


{'accuracy': 0.956140350877193, 'balanced_accuracy': np.float64(0.9454365079365079), 'f1_macro': 0.9521289997480473, 'roc_auc': np.float64(0.9927248677248677), 'log_loss': 0.0945790283840955}
0.2761082130000432
0.004522744000496459
9.125
catboost
{'accuracy': 0.9649122807017544, 'balanced_accuracy': np.float64(0.9573412698412699), 'f1_macro': 0.9619111259605746, 'roc_auc': np.float64(0.9950396825396826), 'log_loss': 0.0897700356685694}
31.152437500000815
0.0011827170001197373
9.125


In [6]:
for model in ["tabdpt_v1.3"]:
    print(model)
    
    result = evaluate(
        model_name=model,
        data=data,
        device="cuda",
    )


    print(result.metrics)
    print(result.fit_seconds)
    print(result.predict_seconds)
    print(result.peak_gpu_memory_mb)

tabdpt_v1.3


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


tabdpt1_3.safetensors: reconstructing file:   0%|          |  0.00B /  252MB            

tabdpt1_3.safetensors: downloading bytes:           |  0.00B            

W0921 18:42:37.292000 40592 torch/_inductor/utils.py:1731] [4/0_1] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.13/dist-packages/tabdpt/model.py:291: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:986.)
  attn = F.scaled_dot_product_attention(q, k, v, scale=custom_scale).transpose(1, 2)
/usr/local/lib/python3.13/dist-packages/tabdpt/model.py:291: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:552.)
  attn = F.scaled_dot_product_attention(q, k, v, scale=custom_scale).transpose(1, 2)
/usr/local/lib/python3.13/dist-packages/tabdpt/model.py:291: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:988.)
  attn = F.scaled_dot_product_attention(q, k, v, scale=custom_scale).transpose(1, 2

RuntimeError: No available kernel. Aborting execution.

In [15]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/talent_data
!unzip -q '/content/drive/talent/benchmark_dataset.zip -d /content/talent_data
!find /content/talent_data -maxdepth 2 -type f | head -50

MessageError: [dfs_ephemeral] Credentials propagation unsuccessful

In [11]:
DATASETS = [
    "Higgs",
    "poker-hand",
    "Data_Science_for_Good_Kiva_Crowdfunding",
    "covertype",
]
data = load_talent_dataset(
    "Higgs",
    root="/content/data",
)

print(data.name)
print(data.X_train.shape)
print(data.X_test.shape)
print(np.unique(data.y_train, return_counts=True))

AssertionError: 

In [10]:
results = benchmark_models(
    models=["xgboost", "catboost", "tabicl_v1", "tabicl_v1.1", "tabicl_v2", "tabpfn_v2", "tabpfn_v2.5", "tabpfn_v2.6", "tabpfn_v3", "tabpfn_v3.5", "tabpfn_v3.5_fast"],
    data=data,
    device="cuda",
    tabpfn_token=TABPFN_TOKEN,
    model_kwargs={
        "xgboost": {
            "n_estimators": 500,
            "max_depth": 8,
        },

        "catboost": {
            "iterations": 500,
            "depth": 8,
        },

        "tabdpt_v1.3": {
            "context_size": 8192,
            "n_ensembles": 1,
        },

        "tabpfn_v3": {
            "n_estimators": 8,
        },
    },
)

NameError: name 'data' is not defined